# Problem 044:  Pentagon Numbers

> Pentagonal numbers are generated by the formula, $P_n=n(3n-1)/2$. The first ten pentagonal numbers are:
> $$1, 5, 12, 22, 35, 51, 70, 92, 117, 145, \dots$$
>
> It can be seen that $P_4 + P_7 = 22 + 70 = 92 = P_8$. However, their difference, $70 - 22 = 48$, is not pentagonal.
>
> Find the pair of pentagonal numbers, $P_j$ and $P_k$, for which their sum and difference are pentagonal and $D = |P_k - P_j|$ is minimised; what is the value of $D$?

## First pass Solution $\mathcal O \left( M^2 \right)$

I will admit that the first code I wrote for this right now took 15 minutes to complete.  I can absolutely see myself, when I originally did the problem, taking the win of getting the answer to the problem and moving on with little to no more thought about it.  One nice aspect about doing this project is that it is making me be more honest and doing these problems with a little more rigor.

Here is a solution that works, and gives the anser in well under a second.  It does not allow for extending the problem to larger values (how many $D<10^12$ are there?), but it is a good first step (or second, if I count the zeroeth order code).

I will use the following indexing:
$$P_m = P_j - P_k,$$
$$P_n = P_j + P_k.$$
where the final value we are looking for is $P_m$.  Also, observe that:
$$P_{k+a} = P_k + P_a + 3ka$$
If $j = k+a$, then:
$$P_m = P_a + 3 k a \implies P_m - P_a = 3ka$$
This last equation suggests the following approach.  Loop through $m$, this way the first value we find that satisfies the constraints will necessarily be the minimum value.  The gap between $k$ and $j$, namely $a$, must be strictly less than $m$.  Also, $P_m \% 3 = P_a \% 3$ in order for the difference to be divisible by $3$, and occurs when $m \% 3 = a \% 3$.  To find possible $j$ and $k$ for a given $m$, loop over all $a < m$, stepping by three.  Then a trial division of $(P_m - P_a) / 3$ by $a$ is done to determine if there exists a $k$ for that specific combination of $m$ and $a$.  If there is, then $k$ and $j$ are determined, and the value $P_j + P_k$ calculated.  If this value is itself a pentagonal number, then a solution is found and $P_m$ is returned.

As for determining if a number is pentagonal or not, invert the equation for the pentagonal numbers:
$$n = \frac{1}{6}\left[1 + \sqrt{1+24P_n}\right].$$
This equation shows that a number $x$ is pentagonal if $1+24x$ is a perfect square and $1+\sqrt{1+24x}$ is divisible by 6.

The Babylonian method (Wikipedia: [Babylonian method](https://en.wikipedia.org/wiki/Square_root_algorithms#Heron's_method)) is used in `is_sqr` to find the closest integer to $\sqrt x$ and test if it is a perfect square.  It returns $0$ if it is not a square, otherwise it returns the square root.

For time scaling, I consider the cost of the calculation searching up to a value $m < M$.  For each value of $m$, $\mathcal O(m)$ values of $a$ are searched.  An overwhelming number of the $m$-$a$ combinations are thrown out due to the divisibility test, so I put the scaling at $\mathcal O(M^2)$.

In [359]:
%%time

def is_sqr(x: int) -> int:
    str_x = str(x)
    y = int(str_x[:len(str_x) // 2 + 1])
    while abs(y - x // y) > 1:
        y = (y + x // y) // 2
    if x == y**2:
        return y
    return 0
    
def is_pent(x: int) -> bool:
    #y = is_sqr(1+24*x)
    #return (y and not (y+1) % 6)
    return (is_sqr(1 + 24 * x) + 1) % 6 == 0

def p044_slow(tot: int) -> int:
    m = 4
    Pm = (m * (3 * m - 1)) // 2
    cnt = 0
    while True:
        a = (m - 1) % 3 + 1
        Pa = (a * (3 * a - 1)) // 2
        while a < m:
            if (Pm - Pa) % (3 * a) == 0:
                k = (Pm - Pa) // a // 3
                Pk = (k * (3 * k - 1)) // 2
                Pn = 2 * Pk + Pa + 3 * k * a
                if is_pent(Pn):
                    cnt += 1
                    if cnt == tot:
                        return Pm
            Pa += 12 + 9 * a
            a += 3
        Pm += 3 * m + 1
        m += 1
    return 0

ans = p044_slow(1)

print(ans)

5482660
CPU times: user 164 ms, sys: 1.01 ms, total: 165 ms
Wall time: 162 ms


## Faster Solution $\mathcal O \left( M \ln M \right)$

A little more derivation and a good bit more support code makes for a much more efficient algorithm.  The slow part of the code above is in searching over all the possible values of $a$ for a given $m$, despite the majority not yielding a possible $k$.  So, let's develop a more direct method of finding $a$ and $k$.

Starting with the relation shown above:
$$P_m - P_a = 3ka \implies 0 = 3 a^2 + (6k-1) a - 2P_m$$
This second relation can be used to solve for $a$,
$$a = \frac{1}{6}\left[-(6k-1) + \sqrt{(6k - 1)^2 + 24 P_m}\right]$$
To have the radical yield an integer we need to solve the Diophantine equaiton:
$$y^2 - x^2 = 24P_m,$$
where $x = (6k - 1)$ and $y$ is resultant of the radical.  The solutions of this equation are found by considering all pairwise factorizations of $24P_m = \alpha\beta$, and having $y = (\beta + \alpha) / 2$ and $x = (\beta - \alpha) / 2$ when $\alpha$ and $\beta$ have the same parity.

For a given factorization of $24P_m$, we then have
$$a = \frac{\alpha}{6},$$
which shows that the smaller of the two factors must be a multiple of $6$.  The same result can be found by finding all the pairwise factorizations of $4P_m$ where $6\alpha \le \beta$.  Better yet, since the two factors need to have the same parity and $6\alpha$ is even, it is the same of finding all the pairwise factors of $2P_m$ where $6\alpha \le 2\beta$. 

Also, in this scheme, recall that $x = 6k-1 = \beta - 3\alpha$, and for $k$ to be an integer, we must also check that $\beta - 3\alpha + 1$ is divisible by $6$.

Ok, so taking a step back, let's turn this information into code.  We will loop through $m$, as before.  All pairwise factorizations of $2P_m = \alpha\beta$ are found where $3\alpha \le \beta$ and $\beta - 3\alpha + 1$ is divisible by $6$.  These restrictions yield all the values of $j$ and $k$ where $P_m = P_j - P_k$.  Then the same code as above is used to test whether $P_j+P_k$ is pentagonal.

The switch of searching over all possible values of $a<m$ in the first code to searching over all pairwise factorizations of $2P_m$ changes the time complexity for a given $m$ from $\mathcal O(m)$ to $\mathcal O(\ln m)$.  Thus the overall time complexity becomes $\mathcal O(M\ln M)$, extending the use of this algorithm to much larger values of $M$.

There are a lot of moving pieces - so here are the functions broken out.

### Prime generation

For quick factorization I use the sieve populated with the smallest prime factor of each value generated from finding primes.  With an open-ended question such as "find the first", there is no clear cut upper limit needed for the prime generation.  In this case, I use my dynamically grown prime generator which extends to cover any newly needed values.

In [360]:
def extend_sieve(pow2: int, primes: list[int]) -> list[int]:
    s = [None] * pow2
    # seive previous primes
    for p in primes[1:]:
        if p * p > 4*pow2:
            break
        i0 = ((2 * pow2) // p + 1) * p
        if i0 % 2 == 0:
            i0 += p
        i0 = (i0 - 2*pow2) // 2
        s[i0::p] = [p if a is None else a for a in s[i0::p]]
    # search for new primes
    for i in range(pow2):
        if s[i] is None:
            s[i] = 2*(pow2+i)+1
            primes.append(2*(pow2+i)+1)
    return s


### Prime factorization

To efficiently find the divisors of a number, its prime factorization is found.  Using the smallest prime factors array `s`, a count of each prime factor in the value is found and added to a dictionary `d` containing the prime and the count.

In [361]:
def get_prime_factors(x: int, d: dict, primes: list[int], s: list[int]) -> None:
    pow2 = len(s) + 1
    if x % 2 == 0:
        x //= 2
        n = 1
        while x % 2 == 0:
            x //= 2
            n += 1
        if 2 in d:
            d[2] += n
        else:
            d[2] = n
    while x > 2 * pow2:
        s += extend_sieve(pow2, primes)
        pow2 *= 2

    while x != 1:
        p = s[x//2-1]
        x //= p
        n = 1
        while x != 1 and s[x//2-1] == p:
            x //= p
            n += 1
        if p in d:
            d[p] += n
        else:
            d[p] = n
    return

### Generate divisors

The number we need to find the divisors of is $2P_m = m(3m - 1)$.  Since we know that $2P_m$ is itself a product of $m$ and $3m-1$, there is no point in sending $2P_m$ to be factored itself, but rather to factor $m$ and $3m-1$ separately and add the results.  This greatly reduces the number of primes needed in the code and is more efficient.  The code then proceeds to find all the divisors by populating the powers of the primes to be less than or equal to those found in the number itself.  Note that this code does not return all the divisors, but only those such that $3\alpha \le \beta$, as discussed above.

In [362]:
from math import prod

def get_divisors(x: int, primes: list[int], s: list[int]) -> None:
    d = dict()
    get_prime_factors(x, d, primes, s)
    get_prime_factors(3*x-1, d, primes, s)

    y = x * (3 * x - 1)
    
    p, tot = zip(*sorted(d.items()))

    np = len(p)
    
    cnt = [0] * np
    i = 0
    while True:
        a = prod(pow(pnow,n) for pnow,n in zip(p, cnt))
        b = y // a
        if 3*a <= b:
            yield a, b
            i = 0
        else:
            for j in range(i+1):
                cnt[i] = 0
            i += 1
        while i < np and cnt[i] == tot[i]:
            cnt[i] = 0
            i += 1
        if i == np:
            break
        cnt[i] += 1
    return

In [363]:
%%time

def is_sqr(x: int) -> int:
    #y = x // 2
    str_x= str(x)
    y = int(str_x[:len(str_x) // 2 + 1])
    while abs(y - x // y) > 1:
        y = (y + x // y) // 2
    if x == y**2:
        return y
    return 0
    
def is_pent(x: int) -> bool:
    return (is_sqr(1+24*x)+1) % 6 == 0

def p044(tot: int) -> int:
    primes = [2]
    s = []

    m = 1
    cnt = 0
    while True:
        for a,b in get_divisors(m, primes, s):
            x = b - 3 * a
            if x % 6 == 5:
                k = (x + 1) // 6
                Pn = (k * (3 * k - 1)) + (a * (3 * a - 1)) // 2 + 3 * k * a

                if is_pent(Pn):
                    cnt += 1
                    #print(m, (m * (3 * m - 1)) // 2)
                    if cnt == tot:
                        return (m * (3 * m - 1)) // 2
        
        m += 1
    return 0

N = 1
ans = p044(N)

print(ans)

5482660
CPU times: user 129 ms, sys: 5 µs, total: 129 ms
Wall time: 128 ms
